<a href="https://colab.research.google.com/github/r-yv/PromptEng/blob/main/Self_Relfection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║   ShopEase Customer Support — Full Menu + ReACT Self-Reflection System       ║
║   Tool: Python (Standard Library) — Google Colab Compatible                  ║
╚══════════════════════════════════════════════════════════════════════════════╝

Iteration history:
  Iteration 1 — Basic 4-step prompt chain, demo cases only, no menu
  Iteration 2 — Interactive main menu + sub-menus added, standard chain
  Iteration 3 — ReACT 6-stage loop added for Shipping & Delivery category
  Iteration 4 — Self-reflection module added (triggered on rating <= 2)
  Iteration 5 (this file) — All above unified: full main menu routes every
                            category through the standard chain; Shipping &
                            Delivery routes through ReACT + self-reflection;
                            rating survey shown after every resolved issue.

SCENARIO (self-reflection trigger):
  Customer selects Shipping & Delivery → types a question about Europe.
  AI returns U.S.-only policy without any geographic disclaimer.
  Customer rates 1/5 → self-reflection module activates, producing:
    • BEFORE summary (what the AI did)
    • Self-critique against 6-criterion rubric
    • AFTER improved summary + customer-facing recovery message
"""

import re
import sys
import textwrap
from dataclasses import dataclass, field
from typing import Optional, List

W = 66   # display width


# ══════════════════════════════════════════════════════════════════════════════
# MOCK DATA
# ══════════════════════════════════════════════════════════════════════════════

ORDER_DB = {
    "ORD-1001": {"status": "shipped",    "item": "Wireless Headphones",
                 "tracking": "UPS-4839201XZ", "carrier": "UPS",
                 "shipped_date": "2026-02-24", "est_delivery": "2026-03-01"},
    "ORD-1002": {"status": "processing", "item": "Mechanical Keyboard",
                 "tracking": None, "carrier": None,
                 "shipped_date": None, "est_delivery": "2026-03-04"},
    "ORD-1003": {"status": "delivered",  "item": "4K Webcam",
                 "tracking": "FDX-0029184", "carrier": "FedEx",
                 "shipped_date": "2026-02-18", "est_delivery": "2026-02-21"},
    "ORD-1004": {"status": "cancelled",  "item": "USB-C Hub",
                 "tracking": None, "carrier": None,
                 "shipped_date": None, "est_delivery": None},
}

# NOTE: "international" key is intentionally absent — this causes the
# European shipping failure that triggers the self-reflection module.
POLICIES = {
    "returns":       "30-day returns for unused items in original packaging. Free return shipping for defective goods.",
    "refunds":       "Refunds issued within 3-5 business days to the original payment method.",
    "shipping":      "Free standard shipping (5-7 days) on orders over $50. Express 2-day: $9.99.",
    "cancellation":  "Orders cancellable within 1 hour of placement only.",
    "damaged":       "Damaged item? Contact us within 48 hours with a photo for free replacement or refund.",
    "warranty":      "All electronics include a 1-year manufacturer warranty covering defects.",
    "delivery_time": "Standard delivery: 5-7 business days from dispatch. Express: 2 business days.",
    "tracking":      "A tracking email is sent within 24 hours of dispatch. Use the carrier website for live updates.",
    "missing_pkg":   "Package not arrived? Contact us within 7 days — we will investigate with the carrier.",
}

NEEDS_ORDER_ID = {"order_status", "returns_refunds", "product_issue", "cancellation"}

GOODWILL_OPTIONS = [
    "A direct email to our international sales team at international@shopease.com",
    "A callback from a specialist within 1 business day to discuss your shipping options",
    "A store credit voucher (10% of cart value) toward a future order once international shipping is available",
]

RATING_LABELS = {
    1: "Very Dissatisfied",
    2: "Dissatisfied",
    3: "Neutral",
    4: "Satisfied",
    5: "Very Satisfied",
}

# ── Menu structure ─────────────────────────────────────────────────────────────
MENU = {
    "1": {
        "label": "Order Status & Tracking",
        "category": "order_status",
        "sub": {
            "1": "Where is my order? It hasn't arrived yet.",
            "2": "My order shows as delivered but I didn't receive it.",
            "3": "I need my tracking number.",
            "4": "My order is still processing — when will it ship?",
        },
    },
    "2": {
        "label": "Returns & Refunds",
        "category": "returns_refunds",
        "sub": {
            "1": "I want to return an item and get a refund.",
            "2": "I returned my item — where is my refund?",
            "3": "I received the wrong item and want to exchange it.",
            "4": "What is your return policy?",
        },
    },
    "3": {
        "label": "Damaged or Defective Item",
        "category": "product_issue",
        "sub": {
            "1": "My item arrived damaged or broken.",
            "2": "My item stopped working after a short time.",
            "3": "I received the wrong item entirely.",
            "4": "My item is missing parts or accessories.",
        },
    },
    "4": {
        "label": "Order Cancellation",
        "category": "cancellation",
        "sub": {
            "1": "I want to cancel my order — I just placed it.",
            "2": "I want to cancel but my order is already processing.",
            "3": "I ordered the wrong item and need to cancel.",
        },
    },
    "5": {
        "label": "Shipping & Delivery  [ReACT + Self-Reflection]",
        "category": "shipping_info",
        "sub": {
            "1": "Type your shipping or delivery question freely.",
        },
    },
    "6": {
        "label": "Billing & Payments",
        "category": "payment",
        "sub": {
            "1": "I was charged incorrectly for my order.",
            "2": "My payment was declined.",
            "3": "I need a copy of my invoice.",
            "4": "I want to update my payment method.",
        },
    },
    "7": {
        "label": "Something Else",
        "category": "general",
        "sub": {
            "1": "I have a question not listed above.",
        },
    },
}


# ══════════════════════════════════════════════════════════════════════════════
# SHARED STATE DATACLASSES
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Context:
    """Standard 4-step chain state — used for all non-shipping categories."""
    raw_message:        str
    category:           Optional[str]  = None
    sentiment:          Optional[str]  = None
    priority:           Optional[str]  = None
    order_id:           Optional[str]  = None
    customer_name:      Optional[str]  = None
    order_record:       Optional[dict] = None
    policy_text:        Optional[str]  = None
    missing_info:       Optional[str]  = None
    resolution:         Optional[str]  = None
    reply:              Optional[str]  = None
    qa_passed:          bool           = False
    qa_notes:           List[str]      = field(default_factory=list)
    escalate:           bool           = False
    final_reply:        Optional[str]  = None
    satisfaction_rating: Optional[int] = None


@dataclass
class ReACTContext:
    """6-stage ReACT state + self-reflection — used for Shipping & Delivery."""
    raw_message:         str
    # Stage 1
    intent:              Optional[str]  = None
    sentiment:           Optional[str]  = None
    priority:            Optional[str]  = None
    order_id:            Optional[str]  = None
    customer_name:       Optional[str]  = None
    requires_order:      bool           = False
    shipping_topics:     List[str]      = field(default_factory=list)
    is_international:    bool           = False
    # Stage 2
    action_plan:         List[str]      = field(default_factory=list)
    # Stage 3
    order_record:        Optional[dict] = None
    policy_snippets:     List[str]      = field(default_factory=list)
    tools_called:        List[str]      = field(default_factory=list)
    # Stage 4
    observations:        List[str]      = field(default_factory=list)
    gaps:                List[str]      = field(default_factory=list)
    needs_fix:           bool           = False
    # Stage 5
    fix_notes:           List[str]      = field(default_factory=list)
    clarify_message:     Optional[str]  = None
    # Stage 6
    qa_notes:            List[str]      = field(default_factory=list)
    qa_passed:           bool           = False
    escalate:            bool           = False
    final_reply:         Optional[str]  = None
    # Self-Reflection
    satisfaction_rating: Optional[int]  = None
    original_summary:    Optional[str]  = None
    critique_results:    List[dict]     = field(default_factory=list)
    improved_summary:    Optional[str]  = None
    diff_notes:          List[str]      = field(default_factory=list)
    recovery_message:    Optional[str]  = None


# ══════════════════════════════════════════════════════════════════════════════
# DISPLAY HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _div(char="-"):
    print(f"  {char * (W - 4)}")

def _div_eq():
    print(f"  {'=' * (W - 4)}")

def _banner(stage: str, label: str, char="#"):
    print(f"\n  {char * (W - 4)}")
    print(f"  {stage} — {label}")
    print(f"  {char * (W - 4)}")

def _field(key: str, value, indent: int = 4):
    prefix = " " * indent
    label  = f"{key:<26}"
    val    = str(value) if value is not None else "—"
    line   = f"{prefix}{label}: {val}"
    if len(line) > W + 4:
        print(f"{prefix}{label}:")
        for part in textwrap.wrap(val, width=W - indent):
            print(f"{' ' * (indent + 2)}{part}")
    else:
        print(line)

def _bullet(text: str, marker: str = "*", indent: int = 6):
    prefix = " " * indent + marker + " "
    lines  = textwrap.wrap(text, width=W - indent - 3)
    print(prefix + (lines[0] if lines else ""))
    for ln in lines[1:]:
        print(" " * (indent + 2) + ln)

def _safe_input(prompt: str) -> str:
    try:
        return input(prompt).strip()
    except (EOFError, KeyboardInterrupt):
        print("\n\n  Session ended. Goodbye!")
        sys.exit(0)

def _print_support_reply(reply: str, label: str = "SUPPORT REPLY",
                          escalate: bool = False, qa_notes: List[str] = None):
    print(f"\n  {'=' * (W - 4)}")
    print(f"  {label}")
    print(f"  {'=' * (W - 4)}")
    plain = (reply or "").replace("**", "").replace("`", "")
    for line in plain.split("\n"):
        for wrapped in (textwrap.wrap(line, width=W - 4) or [""]):
            print(f"  {wrapped}")
    print(f"  {'=' * (W - 4)}")
    if escalate:
        print("  [Case escalated to senior specialist]")
    if qa_notes:
        for n in qa_notes:
            if "FAIL" in n or "FIXED" in n:
                print(f"  [QA] {n}")


# ══════════════════════════════════════════════════════════════════════════════
# UTILITY FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def _extract_order_id(text: str) -> Optional[str]:
    m = re.search(r'\bord-?\d{3,6}\b', text, re.IGNORECASE)
    if m:
        digits = re.sub(r'[^0-9]', '', m.group(0))
        return f"ORD-{digits}"
    return None

def _extract_name(text: str) -> Optional[str]:
    m = re.search(r'\bmy name is ([A-Z][a-z]+)\b', text, re.IGNORECASE)
    return m.group(1).title() if m else None

def _detect_sentiment(msg: str) -> str:
    if any(w in msg for w in ["unacceptable","furious","ridiculous","worst",
                               "outraged","scam","terrible","!!!"]):
        return "angry"
    if any(w in msg for w in ["frustrated","annoyed","disappointed","again",
                               "useless","vague","unclear","still waiting"]):
        return "frustrated"
    if any(w in msg for w in ["thank","great","happy","love","appreciate"]):
        return "positive"
    return "neutral"

def _detect_international(msg: str) -> bool:
    return bool(re.search(
        r'\b(europe|european|uk|germany|france|italy|spain|netherlands|'
        r'poland|sweden|denmark|international|abroad|overseas|outside.{0,10}us|'
        r'outside.{0,10}states|non.?us)\b',
        msg, re.IGNORECASE
    ))

def _collect_rating() -> Optional[int]:
    """Show satisfaction survey; return 1-5 or None if skipped."""
    print()
    _div_eq()
    print("  CUSTOMER SATISFACTION SURVEY")
    _div_eq()
    print("  How satisfied were you with the support you received?\n")
    for k, v in RATING_LABELS.items():
        print(f"    {k}.  {v}")
    print()
    while True:
        raw = _safe_input("  Enter your rating (1-5), or press Enter to skip: ")
        if raw == "":
            print("  Survey skipped.")
            return None
        if raw.isdigit() and int(raw) in RATING_LABELS:
            rating = int(raw)
            print(f"\n  Rating recorded: {rating}/5 — {RATING_LABELS[rating]}")
            return rating
        print("  Please enter a number between 1 and 5.")

def _prompt_order_id() -> Optional[str]:
    print()
    print("  We need your order ID to look up your case.")
    print("  It looks like ORD-1001 and is in your confirmation email.")
    print("  Test IDs: ORD-1001 (shipped)  ORD-1002 (processing)")
    print("            ORD-1003 (delivered) ORD-1004 (cancelled)")
    print("  Press Enter to skip.")
    raw = _safe_input("  Order ID > ")
    if not raw:
        return None
    m = re.search(r'\d{3,6}', raw)
    return f"ORD-{m.group(0)}" if m else None


# ══════════════════════════════════════════════════════════════════════════════
# STANDARD 4-STEP CHAIN  (Order Status, Returns, Product Issues, etc.)
# ══════════════════════════════════════════════════════════════════════════════

def _std_step1_classify(ctx: Context) -> Context:
    """
    PROMPT: Classify the customer's issue. Return category, sentiment,
            priority, order_id, customer_name.
    Constraint: angry → urgent. Product issues → high always.
    """
    msg = ctx.raw_message.lower()
    ctx.order_id      = _extract_order_id(ctx.raw_message)
    ctx.customer_name = _extract_name(ctx.raw_message)
    if ctx.category is None:
        if "cancel" in msg:
            ctx.category = "cancellation"
        elif any(w in msg for w in ["broken","damaged","defective","not working"]):
            ctx.category = "product_issue"
        elif any(w in msg for w in ["return","refund","send back","money back"]):
            ctx.category = "returns_refunds"
        elif any(w in msg for w in ["free shipping","shipping cost","express"]):
            ctx.category = "shipping_info"
        elif any(w in msg for w in ["where is","tracking","not arrived"]):
            ctx.category = "order_status"
        elif any(w in msg for w in ["charge","billed","payment","invoice"]):
            ctx.category = "payment"
        else:
            ctx.category = "general"
    ctx.sentiment = _detect_sentiment(msg)
    ctx.priority  = {"angry":"urgent","frustrated":"high",
                     "neutral":"medium","positive":"low"}[ctx.sentiment]
    if ctx.category == "product_issue":
        ctx.priority = "high"
    return ctx

def _std_step2_gather(ctx: Context) -> Context:
    """
    PROMPT: Look up order record and relevant policy. Flag missing info.
    Constraint: never invent data.
    """
    if ctx.order_id:
        ctx.order_record = ORDER_DB.get(ctx.order_id)
    policy_map = {
        "returns_refunds": "returns",
        "product_issue":   "damaged",
        "cancellation":    "cancellation",
        "shipping_info":   "shipping",
        "order_status":    "shipping",
        "payment":         "refunds",
    }
    ctx.policy_text = POLICIES.get(policy_map.get(ctx.category, ""), None)
    if ctx.category in NEEDS_ORDER_ID and not ctx.order_id:
        ctx.missing_info = ("Could you share your order ID? "
                            "It looks like ORD-XXXX and is in your confirmation email.")
    elif ctx.order_id and ctx.order_record is None:
        ctx.missing_info = (f"I couldn't find order {ctx.order_id}. "
                            "Could you double-check the number?")
    else:
        ctx.missing_info = None
    return ctx

def _std_step3_propose(ctx: Context) -> Context:
    """
    PROMPT: Write a warm reply as Alex. Match tone to sentiment.
    Constraint: no invented data. No refund amount promises.
    """
    name     = ctx.customer_name or "there"
    greeting = f"Hi {name},"
    empathy  = {"angry":      "I'm truly sorry — I want to fix this immediately. ",
                "frustrated": "I'm sorry for the trouble — let me sort this out. ",
                "neutral":    "Thanks for reaching out! ",
                "positive":   "Happy to help! "}[ctx.sentiment]

    if ctx.missing_info:
        ctx.resolution = "ask_for_missing_info"
        ctx.reply = (f"{greeting}\n\n{empathy}{ctx.missing_info}\n\n"
                     "Warm regards,\nAlex | ShopEase Customer Support")
        return ctx

    o, oid = ctx.order_record or {}, ctx.order_id or "your order"

    if ctx.category == "order_status":
        s = o.get("status", "unknown")
        if s == "shipped":
            core = (f"Your order {oid} ({o.get('item','')}) shipped via "
                    f"{o.get('carrier','')} on {o.get('shipped_date','')}. "
                    f"Tracking: {o.get('tracking','')}. ETA: {o.get('est_delivery','')}.")
        elif s == "processing":
            core = (f"Your order {oid} is still being prepared in our warehouse. "
                    f"Expected dispatch by {o.get('est_delivery','')}.")
        elif s == "delivered":
            core = (f"Our records show {oid} was delivered. "
                    "Could you check with a neighbour or building reception? "
                    "Reply here if it's still missing.")
        else:
            core = f"Your order {oid} currently has status: {s}."
    elif ctx.category == "returns_refunds":
        core = (f"You're within our 30-day return window for {oid}. "
                "I'll email you a prepaid return label within 24 hours. "
                "Refund processes within 3-5 business days.")
    elif ctx.category == "product_issue":
        core = (f"I'm so sorry about your {oid}. We can send a free replacement "
                "right away or issue a full refund in 3-5 days. Which would you prefer?")
    elif ctx.category == "cancellation":
        if o.get("status") == "processing":
            core = (f"Order {oid} has entered fulfilment and can't be stopped. "
                    "When it arrives, refuse delivery — we'll refund you in 3-5 days.")
        else:
            core = (f"Done! Order {oid} has been cancelled. "
                    "Your refund will appear within 3-5 business days.")
    elif ctx.category == "shipping_info":
        core = ctx.policy_text or "Please visit our Help Centre for shipping details."
    elif ctx.category == "payment":
        core = ("Billing matters need a specialist. I've flagged your case — "
                "our billing team will contact you within 24 hours.")
    else:
        core = ("I'd be happy to help! Could you share a little more detail "
                "so I can point you in the right direction?")

    ctx.reply = (f"{greeting}\n\n{empathy}{core}\n\n"
                 "Warm regards,\nAlex | ShopEase Customer Support")
    return ctx

def _std_step4_qa(ctx: Context) -> Context:
    """
    PROMPT: Review draft against 5 QA rules. Auto-fix sign-off.
    Hard rule: angry/urgent → append human escalation notice.
    """
    draft, notes = ctx.reply or "", []
    if not re.search(r'\b(hi|hello)\b', draft, re.IGNORECASE):
        notes.append("QA-1 FAIL: Missing greeting.")
    if ctx.order_id and ctx.order_id not in draft:
        notes.append(f"QA-2 FAIL: {ctx.order_id} not in reply.")
    if not re.search(r'\b(will|can|email|send|refund|ship|contact|reply|check)\b',
                     draft, re.IGNORECASE):
        notes.append("QA-3 FAIL: No action verb.")
    if "support" not in draft.lower():
        draft += "\n\nWarm regards,\nAlex | ShopEase Customer Support"
        notes.append("QA-4 AUTO-FIXED: Sign-off added.")
    if len(draft.split()) < 20:
        notes.append("QA-5 FAIL: Reply too short.")
    ctx.qa_passed = not any("FAIL" in n for n in notes)
    ctx.qa_notes  = notes
    if ctx.sentiment == "angry" or ctx.priority == "urgent":
        ctx.escalate = True
        draft += ("\n\n-------------------------------\n"
                  "Your case has been flagged for a senior specialist "
                  "who will contact you within 2 business hours.")
    ctx.final_reply = draft
    return ctx

def run_standard_chain(message: str, category: str = None,
                        customer_name: str = None) -> Context:
    ctx = Context(raw_message=message)
    if category:
        ctx.category = category
    if customer_name:
        ctx.customer_name = customer_name
    for fn in [_std_step1_classify, _std_step2_gather,
               _std_step3_propose, _std_step4_qa]:
        ctx = fn(ctx)
    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# ReACT TOOLS  (Stage 3 — ACT)
# ══════════════════════════════════════════════════════════════════════════════

def tool_lookup_order(order_id: str) -> Optional[dict]:
    """TOOL: order_lookup — returns order dict or None. Never fabricates."""
    return ORDER_DB.get(order_id)

def tool_fetch_policy(topic: str) -> Optional[str]:
    """TOOL: policy_fetch — returns policy string or None. Never invents."""
    return POLICIES.get(topic)

def tool_detect_shipping_topics(msg: str) -> List[str]:
    """TOOL: topic_detector — returns list of relevant shipping sub-topics."""
    topics = []
    if re.search(r'\b(cost|price|fee|how much|free)\b',                    msg): topics.append("shipping")
    if re.search(r'\b(how long|days|time|when will|arrive)\b',             msg): topics.append("delivery_time")
    if re.search(r'\b(track|tracking number|where is my)\b',               msg): topics.append("tracking")
    if re.search(r'\b(missing|lost|not arrived|never received)\b',         msg): topics.append("missing_pkg")
    if re.search(r'\b(international|abroad|overseas|europe|european|uk)\b',msg): topics.append("international")
    return topics if topics else ["shipping"]


# ══════════════════════════════════════════════════════════════════════════════
# ReACT 6-STAGE CHAIN  (Shipping & Delivery)
# ══════════════════════════════════════════════════════════════════════════════

def react_stage1_reason(ctx: ReACTContext) -> ReACTContext:
    """
    PROMPT: Read the message. Determine intent, sentiment, whether order data
    is needed, and which shipping topics apply. Detect international intent.
    Constraint: reason only — no tool calls yet.
    """
    _banner("STAGE 1", "REASON")
    msg = ctx.raw_message.lower()
    ctx.order_id         = _extract_order_id(ctx.raw_message)
    ctx.customer_name    = _extract_name(ctx.raw_message) or ctx.customer_name
    ctx.shipping_topics  = tool_detect_shipping_topics(msg)
    ctx.is_international = _detect_international(msg)

    if   re.search(r'\b(where is|not arrived|missing|lost|never received)\b', msg):
        ctx.intent, ctx.requires_order = "locate_or_investigate_missing", True
    elif re.search(r'\b(track|tracking)\b', msg):
        ctx.intent, ctx.requires_order = "get_tracking_information", True
    elif re.search(r'\b(how long|when will|days|delivery time)\b', msg):
        ctx.intent, ctx.requires_order = "ask_about_delivery_timeframe", bool(ctx.order_id)
    elif re.search(r'\b(cost|price|fee|free|how much)\b', msg):
        ctx.intent, ctx.requires_order = "ask_about_shipping_cost", False
    elif ctx.is_international:
        ctx.intent, ctx.requires_order = "ask_about_international_shipping", False
    else:
        ctx.intent, ctx.requires_order = "general_shipping_question", False

    ctx.sentiment = _detect_sentiment(msg)
    ctx.priority  = {"angry":"urgent","frustrated":"high",
                     "neutral":"medium","positive":"low"}[ctx.sentiment]
    print()
    _field("Message",             ctx.raw_message[:58] + ("..." if len(ctx.raw_message)>58 else ""))
    _field("Intent",              ctx.intent)
    _field("International query", "YES" if ctx.is_international else "no")
    _field("Sentiment",           ctx.sentiment)
    _field("Priority",            ctx.priority)
    _field("Order ID",            ctx.order_id or "none")
    _field("Shipping topics",     ", ".join(ctx.shipping_topics))
    return ctx

def react_stage2_plan(ctx: ReACTContext) -> ReACTContext:
    """
    PROMPT: Build an ordered action plan. Decide which tools to call.
    Constraint: plan only — no tool calls yet.
    """
    _banner("STAGE 2", "PLAN")
    plan = []
    for topic in ctx.shipping_topics:
        plan.append(f"CALL tool_fetch_policy('{topic}')")
    if ctx.requires_order and ctx.order_id:
        plan.append(f"CALL tool_lookup_order('{ctx.order_id}')")
    elif ctx.requires_order:
        plan.append("SKIP tool_lookup_order — no order ID found in message")
        plan.append("FLAG gap: order ID missing, will request in Stage 5")
    if ctx.is_international:
        plan.append("NOTE: international intent — check if 'international' policy exists in DB")
    plan += ["STAGE 4: Observe results and check for gaps",
             "STAGE 5: Fix gaps (clarify or adapt)",
             "STAGE 6: Generate final reply (30-130 words, tone-matched)"]
    ctx.action_plan = plan
    print()
    for i, step in enumerate(plan, 1):
        _bullet(f"Step {i}: {step}")
    return ctx

def react_stage3_act(ctx: ReACTContext) -> ReACTContext:
    """
    PROMPT: Execute planned tools. Record name, input, raw output.
    Constraint: call only tools in the plan.
    """
    _banner("STAGE 3", "ACT — Execute Tools")
    print()
    for topic in ctx.shipping_topics:
        result = tool_fetch_policy(topic)
        label  = f"tool_fetch_policy('{topic}')"
        ctx.tools_called.append(label)
        _bullet(label, marker=">>")
        if result:
            ctx.policy_snippets.append(result)
            _bullet(f"Result: {result[:68]}{'...' if len(result)>68 else ''}", marker="  ")
        else:
            _bullet(f"Result: No policy found for '{topic}'", marker="  ")
        print()
    if ctx.order_id:
        result = tool_lookup_order(ctx.order_id)
        ctx.order_record = result
        label  = f"tool_lookup_order('{ctx.order_id}')"
        ctx.tools_called.append(label)
        _bullet(label, marker=">>")
        _bullet(f"Result: {'Found — ' + result['item'] if result else ctx.order_id + ' NOT FOUND'}", marker="  ")
        print()
    return ctx

def react_stage4_observe(ctx: ReACTContext) -> ReACTContext:
    """
    PROMPT: Review tool results. List observations and gaps.
    Constraint: observe only — no fixes yet.
    """
    _banner("STAGE 4", "OBSERVE")
    obs, gaps = [], []
    if ctx.policy_snippets:
        obs.append(f"{len(ctx.policy_snippets)} policy snippet(s) retrieved: {', '.join(ctx.shipping_topics)}.")
    else:
        gaps.append("No policy snippets retrieved.")
    if ctx.is_international:
        has_intl = any("international" in p.lower() or "customs" in p.lower()
                       for p in ctx.policy_snippets)
        if not has_intl:
            gaps.append(
                "Customer asked about EUROPEAN/INTERNATIONAL shipping but only U.S. "
                "policy was retrieved. Reply will be geographically inaccurate without "
                "an explicit U.S.-only scope disclaimer."
            )
        else:
            obs.append("International policy found — sufficient to answer.")
    if ctx.requires_order:
        if not ctx.order_id:
            gaps.append("Intent requires order data but no order ID found in message.")
        elif ctx.order_record is None:
            gaps.append(f"Order '{ctx.order_id}' provided but NOT found in database.")
        else:
            o = ctx.order_record
            obs.append(f"Order {ctx.order_id} found: '{o['item']}', status='{o['status']}'.")
    if ctx.sentiment in ("angry", "frustrated"):
        obs.append(f"Sentiment '{ctx.sentiment}' — reply must open with empathy.")
    ctx.observations, ctx.gaps, ctx.needs_fix = obs, gaps, len(gaps) > 0
    print()
    print("  Observations:")
    for o in obs:
        _bullet(o, marker="[OK]")
    if gaps:
        print()
        print("  Gaps detected:")
        for g in gaps:
            _bullet(g, marker="[!!]")
    else:
        print()
        _bullet("No gaps — sufficient data to proceed.", marker="[OK]")
    return ctx

def react_stage5_fix(ctx: ReACTContext) -> ReACTContext:
    """
    PROMPT: Address each gap. Re-run tools or set clarify_message.
    Constraint: fix only what Stage 4 flagged. Don't pre-emptively change reply.
    """
    _banner("STAGE 5", "FIX — Iterate")
    notes = []
    if not ctx.needs_fix:
        notes.append("No gaps — all data present. Ready for Stage 6.")
        ctx.fix_notes = notes
        print()
        _bullet(notes[0], marker="[OK]")
        return ctx
    print()
    for gap in ctx.gaps:
        if "EUROPEAN" in gap or "geographically inaccurate" in gap:
            # SIMULATED BEFORE STATE: disclaimer intentionally NOT applied
            # so the flawed reply goes out and triggers the self-reflection.
            notes.append(
                "Gap detected: international query + U.S.-only data. "
                "BEFORE simulation: disclaimer NOT added to reply (intentional failure)."
            )
            _bullet("Gap: international query with U.S.-only data.", marker="[!!]")
            _bullet("Simulating BEFORE state — disclaimer not added to reply.", marker="  ")
        elif "no order ID" in gap.lower():
            ctx.clarify_message = ("Could you share your order ID? "
                                   "It looks like ORD-XXXX and is in your confirmation email.")
            notes.append("FIX: no order ID -> clarify_message set.")
            _bullet("FIX: no order ID -> will ask customer in reply.", marker="->")
        elif "not found in database" in gap.lower():
            ctx.clarify_message = (f"I couldn't find {ctx.order_id}. Could you double-check?")
            notes.append(f"FIX: {ctx.order_id} not in DB -> clarify_message set.")
            _bullet(f"FIX: {ctx.order_id} not found -> asking customer to verify.", marker="->")
        else:
            notes.append(f"Gap noted, no automated fix: {gap[:55]}")
            _bullet(f"Gap noted: {gap[:52]}", marker="[?]")
    ctx.fix_notes = notes
    return ctx

def react_stage6_respond(ctx: ReACTContext) -> ReACTContext:
    """
    PROMPT: Compose final reply using all gathered data. Tone-matched.
    If clarify_message set, use that instead. Run QA check.
    Constraint: no invented data. No refund guarantees. 30-130 words.
    """
    _banner("STAGE 6", "RESPOND — Generate + QA")
    name     = ctx.customer_name or "there"
    greeting = f"Hi {name},"
    empathy  = {"angry":      "I'm truly sorry — I want to resolve this right away. ",
                "frustrated": "I'm sorry for the confusion — let me help you. ",
                "neutral":    "Thanks for reaching out! ",
                "positive":   "Happy to help! "}[ctx.sentiment]

    if ctx.clarify_message:
        draft = (f"{greeting}\n\n{empathy}{ctx.clarify_message}\n\n"
                 "Warm regards,\nAlex | ShopEase Customer Support")
    elif ctx.order_record:
        o, oid = ctx.order_record, ctx.order_id
        s = o.get("status", "unknown")
        if s == "shipped":
            core = (f"Your order {oid} ({o.get('item','')}) has shipped via "
                    f"{o.get('carrier','')}. Tracking: {o.get('tracking','')}. "
                    f"Expected delivery: {o.get('est_delivery','')}.")
        elif s == "processing":
            core = (f"Your order {oid} is still being prepared. "
                    f"Expected dispatch by {o.get('est_delivery','')}.")
        elif s == "delivered":
            core = (f"Our records show {oid} was delivered. "
                    "Check with a neighbour? If still missing, reply here.")
        else:
            core = f"Your order {oid} status: {s}."
        draft = (f"{greeting}\n\n{empathy}{core}\n\n"
                 "Warm regards,\nAlex | ShopEase Customer Support")
    elif ctx.policy_snippets:
        # SIMULATED FAILURE: U.S. policy returned without geographic disclaimer
        core = " ".join(ctx.policy_snippets[:2])
        draft = (f"{greeting}\n\n{empathy}"
                 f"Here's how our shipping works: {core}\n\n"
                 "Warm regards,\nAlex | ShopEase Customer Support")
    else:
        draft = (f"{greeting}\n\n{empathy}"
                 "For shipping details, please visit our Help Centre.\n\n"
                 "Warm regards,\nAlex | ShopEase Customer Support")

    notes = []
    if not re.search(r'\b(hi|hello)\b', draft, re.IGNORECASE):
        notes.append("QA-1 FAIL: Missing greeting.")
    if not re.search(r'\b(will|can|email|send|ship|contact|reply|check|track|visit)\b',
                     draft, re.IGNORECASE):
        notes.append("QA-3 FAIL: No action verb.")
    if "support" not in draft.lower():
        draft += "\n\nWarm regards,\nAlex | ShopEase Customer Support"
        notes.append("QA-4 AUTO-FIXED: Sign-off added.")
    wc = len(draft.split())
    if wc < 20:
        notes.append(f"QA-5 FAIL: Reply too short ({wc} words).")

    ctx.qa_notes  = notes
    ctx.qa_passed = not any("FAIL" in n for n in notes)
    if ctx.sentiment == "angry" or ctx.priority == "urgent":
        ctx.escalate = True
        draft += ("\n\n-------------------------------\n"
                  "Your case has been flagged for a senior specialist "
                  "who will contact you within 2 business hours.")
    ctx.final_reply = draft
    print()
    for n in (notes or ["All QA checks passed (geographic accuracy not in QA scope)."]):
        _bullet(n, marker="[OK]" if "AUTO-FIXED" in n or "passed" in n.lower() else "[!!]")
    return ctx

def run_react_chain(message: str, customer_name: str = None) -> ReACTContext:
    """Run the full 6-stage ReACT pipeline for a shipping query."""
    message = str(message).strip()
    if not message:
        raise ValueError("Customer message cannot be empty.")
    ctx = ReACTContext(raw_message=message)
    if customer_name:
        ctx.customer_name = customer_name
    for fn in [react_stage1_reason, react_stage2_plan, react_stage3_act,
               react_stage4_observe, react_stage5_fix, react_stage6_respond]:
        ctx = fn(ctx)
    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# SELF-REFLECTION MODULE  (activated when rating <= 2)
# ══════════════════════════════════════════════════════════════════════════════

SELF_CRITIQUE_PROMPT_TEMPLATE = """
================================================================
SELF-CRITIQUE PROMPT — ShopEase Shipping Query Review
================================================================
You are a QA reviewer evaluating a support interaction that
received a rating of {rating}/5 ({label}) from the customer.

SCENARIO:
  Customer asked about shipping to Europe.
  AI returned U.S.-only policy without stating it was U.S.-only,
  without acknowledging European shipping, and without offering
  any alternative path for the customer.

ORIGINAL REPLY (the interaction to critique):
----------------------------------------------------------------
{original_reply}
----------------------------------------------------------------

CRITIQUE RUBRIC — evaluate ALL six criteria:

SR-R1 ACCURACY   : Did the AI answer the ACTUAL question (European
                   shipping)? FAIL if U.S. policy returned without
                   geographic scope caveat.

SR-R2 CLARITY    : Was "U.S. only" or equivalent explicitly stated?
                   FAIL if no geographic qualifier present.

SR-R3 AUDIENCE   : Did reply address EU context (customs, duties,
                   transit times)? FAIL if none mentioned.

SR-R4 LENGTH     : Summary 4-6 sentences. Recovery 5-7 sentences.
                   FAIL if under 3 or over 8.

SR-R5 FORMATTING : greeting | empathy | core answer |
                   geographic scope statement | next step | sign-off.
                   FAIL if geographic scope statement absent.

SR-R6 GOAL ALIGN : Was customer given actionable next step?
                   FAIL if no contact path or alternative offered.

CONSTRAINTS:
  - Name the exact missing phrase, not generic observations.
  - Internal tone: objective QA reviewer.
  - No legal promises in fix instructions.
  - One finding + one fix per criterion.
================================================================
"""

def _sr_original_summary(ctx: ReACTContext) -> ReACTContext:
    _banner("SR-A", "ORIGINAL SUMMARY  (BEFORE)", char="=")
    intent  = ctx.intent or "general_shipping_question"
    topics  = ", ".join(ctx.shipping_topics)
    gaps_s  = "; ".join(ctx.gaps) if ctx.gaps else "none"
    ctx.original_summary = (
        f"The AI received a shipping query with intent '{intent}' from a customer "
        f"asking about European delivery (international flag: {'YES' if ctx.is_international else 'no'}). "
        f"Sentiment was '{ctx.sentiment}'. Topics detected: {topics}. "
        f"The policy database returned U.S.-only data — no international policy exists. "
        f"Stage 5 detected the geographic mismatch ({gaps_s}) but did NOT apply a fix, "
        f"allowing the U.S.-scoped reply to be sent without any geographic disclaimer."
    )
    print()
    _bullet("BEFORE SUMMARY:", marker="", indent=2)
    for line in textwrap.wrap(ctx.original_summary, width=W - 6):
        print(f"      {line}")
    return ctx

def _sr_critique(ctx: ReACTContext) -> ReACTContext:
    _banner("SR-B", "SELF-CRITIQUE  (Rubric Applied)", char="=")
    prompt_text = SELF_CRITIQUE_PROMPT_TEMPLATE.format(
        rating=ctx.satisfaction_rating,
        label=RATING_LABELS.get(ctx.satisfaction_rating, "Unknown"),
        original_reply=(ctx.final_reply or "")[:300] + "...",
    )
    print()
    _div("-")
    print("  SELF-CRITIQUE PROMPT APPLIED:")
    _div("-")
    for line in prompt_text.split("\n"):
        print(f"  {line}")
    _div("-")
    print("  CRITIQUE RESULTS:")
    _div("-")

    reply   = (ctx.final_reply or "").lower()
    results = []

    # SR-R1
    has_scope = bool(re.search(r'\b(u\.?s\.?|domestic|united states)\b', reply))
    has_eu    = bool(re.search(r'\b(europe|international|abroad|overseas)\b', reply))
    if not has_scope and not has_eu:
        results.append({"criterion":"SR-R1 ACCURACY","result":"FAIL",
            "finding":"U.S. shipping rates (5-7 days, $9.99) returned for a European "
                      "query with no indication these figures are U.S.-only.",
            "fix":"Check ctx.is_international before returning policy; if True, "
                  "prefix reply with a U.S.-scope statement or route to international branch."})
    else:
        results.append({"criterion":"SR-R1 ACCURACY","result":"PASS",
            "finding":"Reply correctly scoped its shipping information.","fix":"No change."})

    # SR-R2
    has_qualifier = bool(re.search(
        r'\b(u\.?s\.?\s*only|domestic only|united states only|us only)\b', reply))
    if not has_qualifier:
        results.append({"criterion":"SR-R2 CLARITY","result":"FAIL",
            "finding":"Zero geographic scope qualifiers in the reply — no phrase like "
                      "'this applies to U.S. orders only' appears.",
            "fix":"Add before policy text: 'Please note: the following applies to "
                  "U.S. domestic orders only.'"})
    else:
        results.append({"criterion":"SR-R2 CLARITY","result":"PASS",
            "finding":"Geographic scope explicitly stated.","fix":"No change."})

    # SR-R3
    has_eu_content = bool(re.search(r'\b(customs|duties|vat|europe|eu|international)\b', reply))
    if not has_eu_content:
        results.append({"criterion":"SR-R3 AUDIENCE","result":"FAIL",
            "finding":"No mention of customs, duties, VAT, or EU carriers — "
                      "reads as if written for a domestic U.S. customer.",
            "fix":"Add one sentence: 'European orders involve customs clearance "
                  "and different transit timelines — please contact our specialist.'"})
    else:
        results.append({"criterion":"SR-R3 AUDIENCE","result":"PASS",
            "finding":"European-specific content addressed.","fix":"No change."})

    # SR-R4
    summary   = ctx.original_summary or ""
    sentences = [s.strip() for s in re.split(r'[.!?]+', summary) if s.strip()]
    count     = len(sentences)
    if count < 3:
        results.append({"criterion":"SR-R4 LENGTH LIMIT","result":"FAIL",
            "finding":f"Summary has {count} sentence(s) — below 4-sentence minimum.",
            "fix":"Expand to 4-6 sentences: intent, data gap, what returned, what missing."})
    elif count > 8:
        results.append({"criterion":"SR-R4 LENGTH LIMIT","result":"FAIL",
            "finding":f"Summary has {count} sentences — above 6-sentence maximum.",
            "fix":"Trim to 4-6 sentences by removing redundant clauses."})
    else:
        results.append({"criterion":"SR-R4 LENGTH LIMIT","result":"PASS",
            "finding":f"Summary has {count} sentences — within range.","fix":"No change."})

    # SR-R5
    has_scope_stmt = bool(re.search(
        r'\b(applies to|u\.?s\.?\s*only|domestic only|not available|international.*contact)\b', reply))
    if not has_scope_stmt:
        results.append({"criterion":"SR-R5 FORMATTING","result":"FAIL",
            "finding":"Required 'geographic scope statement' section absent from reply.",
            "fix":"Insert 'Scope: U.S. domestic orders only.' between core answer and next step."})
    else:
        results.append({"criterion":"SR-R5 FORMATTING","result":"PASS",
            "finding":"All required sections present.","fix":"No change."})

    # SR-R6
    has_next = bool(re.search(
        r'\b(contact|email|call|visit|specialist|international@|reach out)\b', reply))
    if not has_next:
        results.append({"criterion":"SR-R6 GOAL ALIGNMENT","result":"FAIL",
            "finding":"Customer received no actionable path — no contact link, "
                      "no escalation, no alternative. They end the interaction uninformed.",
            "fix":"Close with: 'For international shipping enquiries, please email "
                  "international@shopease.com or reply here for a specialist callback.'"})
    else:
        results.append({"criterion":"SR-R6 GOAL ALIGNMENT","result":"PASS",
            "finding":"Customer given actionable next step.","fix":"No change."})

    ctx.critique_results = results
    print()
    for r in results:
        icon = "[PASS]" if r["result"] == "PASS" else "[FAIL]"
        print(f"\n    {icon}  {r['criterion']}")
        _bullet(f"Finding: {r['finding']}", indent=10, marker="")
        if r["result"] == "FAIL":
            _bullet(f"Fix    : {r['fix']}", indent=10, marker="")

    fails  = sum(1 for r in results if r["result"] == "FAIL")
    passes = sum(1 for r in results if r["result"] == "PASS")
    print()
    verdict = f"[FAIL] — {fails} issue(s)" if fails else "[PASS]"
    _bullet(f"Overall verdict: {verdict}  ({passes} pass, {fails} fail)", marker=">>", indent=4)
    return ctx

def _sr_improve(ctx: ReACTContext) -> ReACTContext:
    _banner("SR-C", "IMPROVED SUMMARY + RECOVERY MESSAGE  (AFTER)", char="=")
    fails      = [r for r in ctx.critique_results if r["result"] == "FAIL"]
    diff_notes = []
    name       = ctx.customer_name or "there"

    for fail in fails:
        diff_notes.append(
            f"[{fail['criterion']}]\n"
            f"  BEFORE: {fail['finding']}\n"
            f"  AFTER : {fail['fix']}"
        )
    ctx.diff_notes = diff_notes

    fail_labels = ", ".join(r["criterion"] for r in fails) if fails else "none"
    ctx.improved_summary = (
        f"Upon review, this interaction failed on {len(fails)} of 6 rubric criteria: "
        f"{fail_labels}. "
        f"The root cause: the customer asked about European shipping, but the AI's "
        f"policy database contained only U.S. domestic data. Stage 5 detected the "
        f"geographic mismatch but did not apply a fix before Stage 6 ran. "
        f"The corrected interaction must explicitly state 'U.S. domestic only', "
        f"acknowledge that European shipping information is unavailable, and provide "
        f"a direct contact path — addressing accuracy, audience, and goal alignment."
    )

    empathy = (
        "We owe you a genuine apology: the response you received applied to U.S. "
        "domestic customers only and did not address your situation in Europe at all."
        if ctx.sentiment in ("frustrated", "angry")
        else
        "Thank you for your patience — your feedback has helped us identify a real gap."
    )
    ctx.recovery_message = (
        f"Hi {name},\n\n"
        f"{empathy} "
        f"To be transparent: our standard shipping policy (5-7 days, free over $50) "
        f"applies to U.S. domestic orders only. We do not currently offer standard "
        f"international shipping to Europe through our website checkout. "
        f"This should have been stated clearly in your first reply — it was not, "
        f"and that is our fault. "
        f"Here is what we can do for you right now: "
        f"(1) {GOODWILL_OPTIONS[0]}, "
        f"(2) {GOODWILL_OPTIONS[1]}, or "
        f"(3) {GOODWILL_OPTIONS[2]}. "
        f"Please reply here or email support@shopease.com and we will follow up "
        f"within 1 business day."
    )

    # Print diff
    print()
    _div("-")
    print("  BEFORE vs AFTER — DIFF NOTES")
    _div("-")
    if diff_notes:
        for note in diff_notes:
            print()
            for line in note.split("\n"):
                print(f"    {line}")
    else:
        print("  No differences — all criteria passed.")

    # Print AFTER summary
    print()
    _div("-")
    print("  IMPROVED SUMMARY  (AFTER):")
    _div("-")
    for line in textwrap.wrap(ctx.improved_summary, width=W - 6):
        print(f"      {line}")

    # Print recovery message
    print()
    _div_eq()
    print("  RECOVERY MESSAGE  (What Should Have Been Said)")
    _div_eq()
    for line in ctx.recovery_message.split("\n"):
        for wrapped in (textwrap.wrap(line, width=W - 4) or [""]):
            print(f"  {wrapped}")
    _div_eq()
    return ctx

def run_self_reflection(ctx: ReACTContext) -> ReACTContext:
    """Run the 3-stage self-reflection module on a completed ReACT session."""
    print()
    _div_eq()
    rating = ctx.satisfaction_rating
    print(f"  SELF-REFLECTION MODULE ACTIVATED  "
          f"(Rating: {rating}/5 — {RATING_LABELS[rating]})")
    _div_eq()
    print()
    print("  Tools used:")
    _bullet("_sr_original_summary — builds BEFORE summary from session state")
    _bullet("_sr_critique         — applies 6-criterion rubric (SR-R1 to SR-R6)")
    _bullet("_sr_improve          — builds AFTER summary + customer recovery message")
    print()
    print("  Requirements applied:")
    _bullet("SR-R1 ACCURACY    — geographic scope of reply matches query")
    _bullet("SR-R2 CLARITY     — 'U.S. only' scope explicitly stated")
    _bullet("SR-R3 AUDIENCE    — European customer context addressed")
    _bullet("SR-R4 LENGTH      — summaries 4-6 sentences; recovery 5-7 sentences")
    _bullet("SR-R5 FORMATTING  — all 6 sections present incl. geographic scope stmt")
    _bullet("SR-R6 GOAL ALIGN  — customer given actionable next step")
    ctx = _sr_original_summary(ctx)
    ctx = _sr_critique(ctx)
    ctx = _sr_improve(ctx)
    return ctx


# ══════════════════════════════════════════════════════════════════════════════
# MENU DISPLAY FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def _show_main_menu():
    print()
    _div_eq()
    print("  SHOPEASE CUSTOMER SUPPORT — MAIN MENU")
    _div_eq()
    print("  What can we help you with today?\n")
    for key, entry in MENU.items():
        tag = "  [ReACT + Self-Reflection]" if entry["category"] == "shipping_info" else ""
        print(f"    {key}.  {entry['label']}{tag}")
    print()
    print("    0.  Exit")
    _div()

def _show_sub_menu(entry: dict):
    print()
    _div()
    print(f"  {entry['label'].upper()}")
    _div()
    print("  Please select your specific issue:\n")
    for key, text in entry["sub"].items():
        lines = textwrap.wrap(text, width=W - 10)
        print(f"    {key}.  {lines[0]}")
        for extra in lines[1:]:
            print(f"         {extra}")
    print()
    print("    0.  Back to main menu")
    _div()

def _closing(escalate: bool = False):
    print()
    _div_eq()
    print("  Thank you for contacting ShopEase Customer Support!")
    if escalate:
        print("  A specialist will follow up with you shortly.")
    print("  Have a great day!")
    _div_eq()


# ══════════════════════════════════════════════════════════════════════════════
# MAIN SESSION  —  full menu + chain routing
# ══════════════════════════════════════════════════════════════════════════════

def run_session():
    """
    Full interactive session:
      1. Welcome + optional name
      2. Main menu → category selection
      3a. Shipping & Delivery → free-text → ReACT chain → rating → self-reflection if <= 2
      3b. All other categories → sub-menu → order ID → extra detail → standard chain → rating
      4. Ask if customer needs more help; loop or close
    """
    print()
    _div_eq()
    print("  Welcome to ShopEase Customer Support")
    print("  Powered by a 4-step chain + 6-stage ReACT reasoning")
    _div_eq()
    print()
    print("  Tip: Choose option 5 (Shipping & Delivery) and ask about")
    print("  shipping to Europe to see the full self-reflection module.")
    print("  Test order IDs: ORD-1001  ORD-1002  ORD-1003  ORD-1004")
    _div()

    raw_name = _safe_input("\n  Your first name (optional, press Enter to skip): ")
    name = raw_name.title() if raw_name else None
    if name:
        print(f"\n  Hi {name}! Let's get your issue sorted.")
    else:
        print("\n  Let's get your issue sorted.")

    while True:
        # ── MAIN MENU ─────────────────────────────────────────────────────────
        _show_main_menu()
        choice = _safe_input("  Choose an option > ")

        if choice == "0":
            print("\n  Thank you for contacting ShopEase. Have a great day!")
            break

        if choice not in MENU:
            print("  Please enter a number shown in the menu.")
            continue

        entry    = MENU[choice]
        category = entry["category"]

        # ════════════════════════════════════════════════════════════════
        # PATH A — Shipping & Delivery: ReACT chain + self-reflection
        # ════════════════════════════════════════════════════════════════
        if category == "shipping_info":
            print()
            _div()
            print("  SHIPPING & DELIVERY  [6-Stage ReACT Reasoning Mode]")
            _div()
            print("  Type your shipping or delivery question below.")
            print("  Include an order ID (e.g. ORD-1001) for order-specific help.")
            print("  Ask about Europe or international shipping to see self-reflection.")
            print()

            while True:
                msg = _safe_input("  You > ")
                if not msg:
                    print("  Please type a message before pressing Enter.")
                    continue
                if msg.lower() in {"back", "menu"}:
                    break

                if name:
                    msg = f"My name is {name}. {msg}"

                print()
                _div()
                print("  Running 6-stage ReACT reasoning loop...")
                _div()

                try:
                    react_ctx = run_react_chain(msg, customer_name=name)
                except ValueError as e:
                    print(f"  Error: {e}")
                    continue

                _print_support_reply(react_ctx.final_reply,
                                     label="SUPPORT REPLY (BEFORE SELF-REFLECTION)",
                                     escalate=react_ctx.escalate,
                                     qa_notes=react_ctx.qa_notes)

                # Survey
                rating = _collect_rating()
                react_ctx.satisfaction_rating = rating

                # Self-reflection if dissatisfied
                if rating is not None and rating <= 2:
                    react_ctx = run_self_reflection(react_ctx)
                elif rating is not None:
                    label = RATING_LABELS[rating]
                    print(f"\n  {label} — thank you for your feedback!")
                    if rating >= 4:
                        print("  Great to hear — we're glad we could help!")

                print()
                again = _safe_input("  Another shipping question? (yes / no) > ").lower()
                if again not in ("yes", "y"):
                    _closing(react_ctx.escalate)
                    return
                # else: loop back for another shipping question

        # ════════════════════════════════════════════════════════════════
        # PATH B — All other categories: standard 4-step chain
        # ════════════════════════════════════════════════════════════════
        else:
            while True:
                _show_sub_menu(entry)
                sub_choice = _safe_input("  Choose your issue > ")

                if sub_choice == "0":
                    break   # back to main menu

                if sub_choice not in entry["sub"]:
                    print("  Please enter a number from the list above.")
                    continue

                sub_text = entry["sub"][sub_choice]

                # Order ID collection for relevant categories
                order_id = None
                if category in NEEDS_ORDER_ID:
                    order_id = _prompt_order_id()

                # Optional extra detail
                print()
                print("  Anything to add? (Press Enter to skip)")
                extra   = _safe_input("  You > ")
                message = f"{sub_text} {extra}".strip() if extra else sub_text
                if name:
                    message = f"My name is {name}. {message}"
                if order_id:
                    message = f"{message} My order ID is {order_id}."

                print()
                _div()
                print("  Processing your request...")
                _div()

                std_ctx = run_standard_chain(message, category=category,
                                              customer_name=name)
                _print_support_reply(std_ctx.final_reply,
                                     escalate=std_ctx.escalate,
                                     qa_notes=std_ctx.qa_notes)

                # Survey
                rating = _collect_rating()
                std_ctx.satisfaction_rating = rating
                if rating is not None:
                    label = RATING_LABELS[rating]
                    if rating <= 2:
                        print(f"\n  We're sorry to hear that ({label}).")
                        print("  Please email support@shopease.com for further assistance.")
                        print("  A specialist will review your case within 1 business day.")
                    else:
                        print(f"\n  {label} — thank you for your feedback!")

                print()
                again = _safe_input("  Anything else we can help with? (yes / no) > ").lower()
                if again not in ("yes", "y"):
                    _closing(std_ctx.escalate)
                    return
                break   # back to main menu for new category


# ══════════════════════════════════════════════════════════════════════════════
# EXECUTION LOG  —  evidence of output (python script.py --log)
# ══════════════════════════════════════════════════════════════════════════════

EXECUTION_LOG_CASES = [
    {
        "title":   "LOG 1 — European shipping query (primary self-reflection scenario)",
        "message": "Hi, how does shipping work if I'm in Europe? Do you ship to Germany?",
        "rating":  1,
        "name":    "Sophie",
        "note":    "AI returns U.S. policy. Rating 1/5 → self-reflection activates.",
    },
    {
        "title":   "LOG 2 — Follow-up: customer tries again after vague reply",
        "message": "Your reply was about U.S. shipping but I asked about Europe. Can you clarify?",
        "rating":  2,
        "name":    "Sophie",
        "note":    "Frustrated sentiment. Rating 2/5 → self-reflection activates.",
    },
    {
        "title":   "LOG 3 — U.S. shipping question (self-reflection should NOT activate)",
        "message": "How long does standard shipping take? I'm in California.",
        "rating":  4,
        "name":    "James",
        "note":    "Policy applies correctly. Rating 4/5 → no self-reflection.",
    },
]

def run_execution_log():
    print(f"\n  {'#' * (W - 4)}")
    print("  EXECUTION LOG — Full ReACT + Self-Reflection Evidence")
    print(f"  {'#' * (W - 4)}")
    for i, case in enumerate(EXECUTION_LOG_CASES, 1):
        print(f"\n\n  {'=' * (W - 4)}")
        print(f"  {case['title']}")
        print(f"  Note: {case['note']}")
        print(f"  {'=' * (W - 4)}")
        try:
            react_ctx = run_react_chain(case["message"], customer_name=case.get("name"))
            _print_support_reply(react_ctx.final_reply,
                                 label="SUPPORT REPLY (BEFORE SELF-REFLECTION)",
                                 escalate=react_ctx.escalate,
                                 qa_notes=react_ctx.qa_notes)
            react_ctx.satisfaction_rating = case["rating"]
            label = RATING_LABELS[case["rating"]]
            print(f"\n  [Simulated rating: {case['rating']}/5 — {label}]")
            if case["rating"] <= 2:
                react_ctx = run_self_reflection(react_ctx)
            else:
                print(f"\n  Rating {case['rating']}/5 ({label}) — self-reflection not activated.")
        except ValueError as e:
            print(f"\n  [Error caught correctly]: {e}")
        if i < len(EXECUTION_LOG_CASES):
            _safe_input("\n  Press Enter for the next log case...")
    print(f"\n  {'#' * (W - 4)}")
    print("  Execution log complete.")
    print(f"  {'#' * (W - 4)}\n")


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    if "--log" in sys.argv:
        run_execution_log()   # non-interactive: runs all 3 evidence cases
    else:
        run_session()         # default: full interactive menu session


  Welcome to ShopEase Customer Support
  Powered by a 4-step chain + 6-stage ReACT reasoning

  Tip: Choose option 5 (Shipping & Delivery) and ask about
  shipping to Europe to see the full self-reflection module.
  Test order IDs: ORD-1001  ORD-1002  ORD-1003  ORD-1004
  --------------------------------------------------------------

  Your first name (optional, press Enter to skip): John

  Hi John! Let's get your issue sorted.

  SHOPEASE CUSTOMER SUPPORT — MAIN MENU
  What can we help you with today?

    1.  Order Status & Tracking
    2.  Returns & Refunds
    3.  Damaged or Defective Item
    4.  Order Cancellation
    5.  Shipping & Delivery  [ReACT + Self-Reflection]  [ReACT + Self-Reflection]
    6.  Billing & Payments
    7.  Something Else

    0.  Exit
  --------------------------------------------------------------
  Choose an option > 5

  --------------------------------------------------------------
  SHIPPING & DELIVERY  [6-Stage ReACT Reasoning Mode]
  -------------